# Staged Sentiment Analysis Training Pipeline
### End-to-End GPU Training: Stage A (Sentiment140) + Stage B (TweetEval 3-Class)

This notebook trains an enterprise-grade 3-class sentiment model (`negative`, `neutral`, `positive`) in two sequential stages:
1. **Stage A — Domain Adaptation**: Fine-tune a base transformer (`google-bert/bert-base-uncased`) on Sentiment140 with enterprise cleaning (HTML unescaping, URL removal, @user standardization) and 80/10/10 splitting.
2. **Stage B — 3-Class Fine-Tuning**: Swap classification head to 3 classes (`negative`, `neutral`, `positive`), fine-tune on CardiffNLP TweetEval.
3. **Stage C — Strict Benchmark Evaluation**: Evaluate on both the Stage A 10% test set and the untouched TweetEval 3-class test set, computing macro-F1, per-class metrics, and confusion matrix.

> **GPU Setup:** Make sure you are using a GPU runtime in Colab: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

In [ ]:
# 2. Clone the repository and install dependencies
!git clone https://github.com/HabtamuShewamene/AIproject.git
%cd AIproject
!git checkout phase/2-data-model-audit

!pip install -q transformers datasets accelerate scikit-learn pandas

## 3. Supply the Raw Sentiment140 Dataset
Mount your Google Drive to copy `training.1600000.processed.noemoticon.csv` to `data/raw/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p data/raw
# Update this path to where your training CSV is stored on your Drive:
!cp /content/drive/MyDrive/training.1600000.processed.noemoticon.csv data/raw/

# Verify file existence and size
!ls -lh data/raw/

## 4. Prepare Datasets (Enterprise Cleaning & 80/10/10 Splitting)
Run the deterministic preparation and split scripts. These scripts apply enterprise text cleaning (HTML unescaping, URL removal, @user normalization, character elongation squashing), split data into strict 80% Train, 10% Validation, and 10% Test sets, prevent split leakage, and generate verifiable SHA-256 manifests.

In [ ]:
# Prepare Sentiment140 with enterprise cleaning & 80/10/10 splitting
!python -m src.prepare_sentiment140

# Download and validate pinned TweetEval 3-class splits (train, val, test)
!python -m src.download_tweet_eval

## 5. Execute Staged Training
- **Stage A** trains `models/stage_a_sentiment140` (binary adaptation on 80% train, monitored on 10% val, evaluated on 10% test).
- **Stage B** trains `models/bert_sentiment` (3-class fine-tuning on TweetEval).

In [ ]:
# Run Stage A (Binary Sentiment140 Adaptation)
!python -m src.train_staged_sentiment --stage a

In [ ]:
# Run Stage B (Three-Class TweetEval Fine-Tuning)
!python -m src.train_staged_sentiment --stage b

## 6. Evaluate and Inspect Metrics

In [ ]:
import json
metrics_path = 'results/staged_training_metrics.json'
with open(metrics_path) as f:
    metrics = json.load(f)
print(json.dumps(metrics, indent=2))

## 7. Package and Export Model
Compress the resulting model directory and save it to Google Drive so you can load it in the local FastAPI application.

In [ ]:
# Create zip archive of final 3-class model
!zip -r bert_sentiment_trained.zip models/bert_sentiment/

# Copy to Google Drive
!cp bert_sentiment_trained.zip /content/drive/MyDrive/
!cp results/staged_training_metrics.json /content/drive/MyDrive/
print("Model saved to Google Drive as bert_sentiment_trained.zip!")